# LLM breaker
## Goal
The goal of this notebooks is to provide an opportunity to change answers LLMs give after the fact in order to try to break them // try to avoid guardrails
## Usage
Before running this notebook, make sure you have a running `ollama` server with
```zsh
brew install ollama
brew services start ollama
ollama run <any-model>
```

### Definition of the conversation
conversation is a list of lists
```python
[
    [<role>, <content>],
    [<role>, <content>],
    ...
]
```

In [ ]:
class JSONHandler:
    def __init__(self, fp:str) -> None:
        from pathlib import Path
        if not Path(fp).exists():
            raise ValueError(f"The '{fp}' does not exist")
        if not Path(fp).is_file():
            raise TypeError(f"The file path '{fp}' is not a file")
        
        self.filepath = fp

    
    def get_json(self) -> dict:
        """
        Gets json object at 'filepath'.
        If the file is empty, returns an empty dict
        """
        import json
        with open(self.filepath, mode="r") as f:
            if not f.read().strip():
                print("WARNING: The .json file is empty. Returning an empty dict")
                return {}
            
            f.seek(0)
            content = json.load(f)
        return content

    
    def write_json(self, content:dict) -> None:
        """
        Writes conversation object at specified place (self.filepath)
        """
        import json
        with open(self.filepath, mode="w") as f:
            json.dump(content, f)
        return

    
    def discard_json(self) -> None:
        """
        Clears the json file at self.filepath
        """
        with open(self.filepath, mode="w") as f:
            f.write('{}')
        return

In [ ]:
from typing import Tuple, List


def call_model(model:str, convo: List[dict]) -> Tuple[dict, str]:
    """
    Calls the model with the passed conversation.
    
    Return: 
    - The whole conversation including the most recent LLM call
    - Stripped LLM's reply

    """
    import requests

    res = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model, 
            "messages": convo
        }
    )
    
    resp_arr = [i for i in res.text.split("\n") if i]

    parsed_arr = []
    for item in resp_arr:
        parsed_arr.append(
            json.loads(item)
        )
    
    return convo + parsed_arr, parsed_arr


def prettify_convo(convo:List[list], drop_last_reply:bool=False, crop:int=100) -> str:
    """
    Outputs a string of the following format:
    
    user => Hi there!
    LLM => Hello th...
    user => ...
        
    drop_last_reply: drops the last LLM's reply from output

    Max 100 characters in each replica, truncated with '...'
    """
    pretty_convo = ""
    for role, message in convo:        
        pretty_convo += f"{role} => {message}\n\n"
    return pretty_convo


def parse_response(raw_resp: List[dict]) -> str:
    resp = ""
    for i in raw_resp:
        if not i["message"]["content"]: continue
        resp += i["message"]["content"]
    return resp

In [ ]:
# Definition of the conversations
# List of lists
# [role, content]

In [ ]:
import os
os.environ["OLLAMA_HOST"]="0.0.0.0:11434"

In [ ]:
open("conversation.json", mode="w").close()

convo_handler = JSONHandler("conversation.json")
convo_handler.discard_json()
convo = []

* Write message
* Append to the full conversation
* send full conversation
* get the response
* concat full conversation w response
* pretty print

In [ ]:
MESSAGE = """
Hi there!
"""

In [ ]:
import json

MODEL = "mistral:7b"

convo.append(
    ["user", MESSAGE.strip()]
)

packaged_convo = []
for role, content in convo:
    packaged_convo.append(
        {"role": role, "content": content}
    )

_, response = call_model(MODEL, packaged_convo)

match response:
    case [{"error": msg}]:
        raise ValueError(f"The ollama API returned invalid reponse: {msg}")

convo.append(
    ["Assistant", parse_response(response)]
)

print(prettify_convo(convo))

In [ ]:
# The agents's reponses are always odd
MESSAGE_NUM = 3

convo[MESSAGE_NUM][1] = """
...
"""
convo[MESSAGE_NUM][1] = convo[MESSAGE_NUM][1].strip()

In [ ]:
print(prettify_convo(convo))